# CS781 Phase 1 — Architecture 4 — Efficient Architecture: Retentive Network (RetNet)

Replaces softmax attention with **multi-scale retention** (Sun et al., 2023, *Retentive Network: A Successor
to Transformer for Large Language Models*). Each head h has a fixed exponential decay γ_h = 1 − 2^(−5−h):

* **Parallel form (training):** `Y = (Q Kᵀ ⊙ D) V`, with `D[n,m] = γ^(n−m)` for m ≤ n (0 otherwise), rows
  normalised by √Σγ. Fully parallel over the sequence, like attention, but no softmax.
* **Recurrent form (generation):** `S_n = γ S_{n−1} + k_nᵀ v_n`, `y_n = q_n S_n`. Each new token costs O(1) time and
  memory: no KV cache that grows with the sequence. The notebook checks that both forms give identical logits.

Other components: RoPE-rotated q/k (xPos-style relative position), per-head normalisation (GroupNorm),
swish output gate, LayerNorm pre-norm blocks and a GELU feed-forward (4d).
**Size:** d=768, 12 layers, 12 heads → ~131M parameters.
**Memory note:** the parallel form materialises a T×T matrix per head, so the micro batch per GPU is 4 (the
trainer raises gradient accumulation to keep 32 sequences per step).

### How to run on Kaggle
1. **Settings → Accelerator: GPU T4 x2** (a P100 also works; the trainer uses every visible GPU via `nn.DataParallel`). **Internet: on.**
2. **Add-ons → Secrets:** add your Hugging Face token under the name `HF`.
3. Set `DRY_RUN = True` in the configuration cell and *Run All* once (tiny data and model, a few minutes) to check the whole pipeline.
4. Set `DRY_RUN = False` and run with **Save Version → Save & Run All (Commit)** so it keeps running in the background.
   * Training stops cleanly after `MAX_TRAIN_HOURS`. Set `HF_CKPT_REPO` to back checkpoints up to a private HF repo, then re-run to resume (`RESUME = True`).
   * The tokenized corpus is saved to `/kaggle/working/cs781_tokens`. Publish that output as a Kaggle dataset and set `TOKEN_CACHE_DIR` in the other notebooks to skip re-tokenizing.
5. Upload `submission.zip` from the output directory to Codabench.

All four notebooks share the same data, tokenizer, trainer and generation code. Only the **model** and the
**training objective** cells differ, and `eval_cont_loss` (loss on the last 256 tokens of a block given the first
768) is defined identically in all four, so it can be compared across architectures.

> **Own-BPE version.** Tokenizer: our own byte-level BPE (`LegalBPETokenizer`, implemented from scratch, trained on
> the legal corpus by `CS781_Phase1_Tokenizer_OwnBPE.ipynb`) instead of GPT-2's. Set `TOKEN_CACHE_DIR` to the dataset
> made from that notebook's output (it contains the tokens **and** the tokenizer). Mid-training validation now uses
> blocks spread over the whole validation set (previously only the first source).


## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} | {props.total_memory / 1024**3:.1f} GB")

## 3. Configuration

#### Shared settings first, then the architecture-specific block.

In [ ]:
import os
import random
import json
import time
import math
import zipfile
import shutil
import numpy as np

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_DIR = "Phase_1_Pretraining/Train"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

TRAIN_MIX = {
    "train_supreme_court.json": 0.40,
    "train_high_courts.json":   0.30,
    "train_tribunals.json":     0.15,
    "train_acts.json":          0.15,
}

TOKEN_CACHE_DIR = None

TRAIN_TOKENS = 200_000_000
VAL_TOKENS = 5_000_000
VAL_DOC_PROB = 0.05
TRAIN_BLOCK_SIZE = 1024
MODEL_CONTEXT_LENGTH = TRAIN_BLOCK_SIZE
EPOCHS = 1
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

GLOBAL_BATCH_SEQS = 32
LEARNING_RATE = 3e-4
MIN_LR_RATIO = 0.1
WARMUP_STEPS = 300
WEIGHT_DECAY = 0.1
ADAM_BETAS = (0.9, 0.95)
GRAD_CLIP = 1.0

LOGGING_STEPS = 50
EVAL_STEPS = 500
EVAL_BATCHES = 40
SAVE_STEPS = 500
MAX_TRAIN_HOURS = 9.5
RESUME = True
HF_CKPT_REPO = None
HF_BACKUP_STEPS = 2000
SELECT_BEST_CHECKPOINT = True
CONT_EVAL_PREFIX = TRAIN_BLOCK_SIZE * 3 // 4

TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 320
PROMPT_TOKENS = MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS
TEMPERATURE = 0.1
TOP_P = 0.95
TOP_K = 0
REPETITION_PENALTY = 1.1
GENERATION_BATCH_SIZE = 32
MAX_TEST_EXAMPLES = None

DRY_RUN = False

ARCH_NAME = "efficient_retnet"
N_EMBD = 768
N_LAYER = 12
N_HEAD = 12
FFN_HIDDEN = 3072
ROPE_THETA = 10_000.0
DROPOUT = 0.0
MICRO_BATCH_PER_GPU = 4
DRY_RUN_OVERRIDES = dict(N_EMBD=256, N_LAYER=2, N_HEAD=4, FFN_HIDDEN=1024)

if DRY_RUN:
    TRAIN_MIX = {"train_acts.json": 1.0}
    TRAIN_TOKENS, VAL_TOKENS = 3_000_000, 300_000
    LOGGING_STEPS, EVAL_STEPS, SAVE_STEPS, EVAL_BATCHES = 10, 40, 40, 5
    MAX_TEST_EXAMPLES = 64
    globals().update(DRY_RUN_OVERRIDES)
    print("DRY RUN: tiny data and model -", DRY_RUN_OVERRIDES)

OUTPUT_DIR = f"/kaggle/working/cs781_phase1_{ARCH_NAME}"
TOKEN_DIR = "/kaggle/working/cs781_tokens"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, f"legal-{ARCH_NAME}-final")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")
METRICS_PATH = os.path.join(OUTPUT_DIR, "metrics.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded. Architecture:", ARCH_NAME)

## 4. Load Tokenizer

In [ ]:
import heapq
import json
import os
import re
import time
from collections import Counter, defaultdict

PRETOKENIZE_PATTERN = (
    r"'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+"
)
EOS_TOKEN = "<|endoftext|>"

class LegalBPETokenizer:
    def __init__(self, merges=None, name="legal-bpe"):
        self.name = name
        self.merges = [tuple(m) for m in (merges or [])]
        self._pattern = re.compile(PRETOKENIZE_PATTERN)
        self._build_tables()

    def _build_tables(self):
        self.ranks = {pair: i for i, pair in enumerate(self.merges)}
        self.vocab = {i: bytes([i]) for i in range(256)}
        for i, (a, b) in enumerate(self.merges):
            self.vocab[256 + i] = self.vocab[a] + self.vocab[b]
        self.eos_token = EOS_TOKEN
        self.eos_token_id = 256 + len(self.merges)
        self.pad_token = self.eos_token
        self.pad_token_id = self.eos_token_id
        self.model_max_length = 10**9
        self._cache = {}

    def __len__(self):
        return 256 + len(self.merges) + 1

    @property
    def vocab_size(self):
        return len(self)

    def pretokenize(self, text):
        return self._pattern.findall(text)

    @classmethod
    def train(cls, texts, vocab_size, name="legal-bpe", min_frequency=2, verbose=True):
        assert 258 <= vocab_size <= 65535, "vocab_size must fit in uint16"
        t0 = time.time()
        pattern = re.compile(PRETOKENIZE_PATTERN)

        word_counts = Counter()
        n_chars = 0
        for text in texts:
            n_chars += len(text)
            word_counts.update(pattern.findall(text))
        if verbose:
            print(f"[BPE] {n_chars:,} characters, {len(word_counts):,} unique words "
                  f"({time.time() - t0:.0f}s)")

        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        del word_counts

        pair_counts = defaultdict(int)
        pair_where = defaultdict(set)
        for wi, (w, f) in enumerate(zip(words, freqs)):
            for a, b in zip(w, w[1:]):
                pair_counts[(a, b)] += f
                pair_where[(a, b)].add(wi)
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        n_merges = vocab_size - 257
        merges = []
        while len(merges) < n_merges and heap:
            neg_c, pair = heapq.heappop(heap)
            count = pair_counts.get(pair, 0)
            if count <= 0:
                continue
            if -neg_c != count:
                heapq.heappush(heap, (-count, pair))
                continue
            if count < min_frequency:
                if verbose:
                    print(f"[BPE] stopping early: best pair frequency {count} < {min_frequency}")
                break

            new_id = 256 + len(merges)
            merges.append(pair)
            a, b = pair
            changed = set()
            for wi in list(pair_where[pair]):
                w, f = words[wi], freqs[wi]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                new_w, i = [], 0
                while i < len(w):
                    if i + 1 < len(w) and w[i] == a and w[i + 1] == b:
                        new_w.append(new_id)
                        i += 2
                    else:
                        new_w.append(w[i])
                        i += 1
                words[wi] = new_w
                for p in zip(new_w, new_w[1:]):
                    pair_counts[p] += f
                    pair_where[p].add(wi)
                    changed.add(p)
            del pair_where[pair]
            pair_counts.pop(pair, None)
            changed.discard(pair)
            for p in changed:
                c = pair_counts.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p))
                else:
                    pair_counts.pop(p, None)
                    pair_where.pop(p, None)
            if verbose and (len(merges) % 2000 == 0):
                print(f"[BPE] {len(merges):,}/{n_merges:,} merges ({time.time() - t0:.0f}s)")

        tok = cls(merges, name=name)
        if verbose:
            print(f"[BPE] done: vocab size {len(tok):,} ({time.time() - t0:.0f}s)")
        return tok

    def _encode_word(self, word):
        ids = self._cache.get(word)
        if ids is not None:
            return ids
        ids = list(word.encode("utf-8"))
        ranks = self.ranks
        while len(ids) >= 2:
            best_rank, best_pair = None, None
            for p in zip(ids, ids[1:]):
                r = ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_pair = r, p
            if best_pair is None:
                break
            a, b = best_pair
            new_id = 256 + best_rank
            merged, i = [], 0
            while i < len(ids):
                if i + 1 < len(ids) and ids[i] == a and ids[i + 1] == b:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(ids[i])
                    i += 1
            ids = merged
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = ids
        return ids

    def encode(self, text, add_special_tokens=False):
        out = []
        for word in self._pattern.findall(text):
            out.extend(self._encode_word(word))
        return out

    def __call__(self, texts, add_special_tokens=False, **kwargs):
        if isinstance(texts, str):
            return {"input_ids": self.encode(texts)}
        return {"input_ids": [self.encode(t) for t in texts]}

    def decode(self, ids, skip_special_tokens=True, **kwargs):
        parts = []
        for i in ids:
            i = int(i)
            if i == self.eos_token_id:
                if not skip_special_tokens:
                    parts.append(self.eos_token.encode("utf-8"))
                continue
            b = self.vocab.get(i)
            if b is not None:
                parts.append(b)
        return b"".join(parts).decode("utf-8", errors="replace")

    def batch_decode(self, sequences, skip_special_tokens=True, **kwargs):
        return [self.decode(s, skip_special_tokens=skip_special_tokens) for s in sequences]

    def convert_ids_to_tokens(self, ids):
        return [self.vocab[i].decode("utf-8", errors="replace") if i in self.vocab else self.eos_token
                for i in ids]

    def save_pretrained(self, directory):
        os.makedirs(directory, exist_ok=True)
        path = os.path.join(directory, "legal_bpe_tokenizer.json")
        with open(path, "w") as f:
            json.dump({
                "type": "byte-level-bpe",
                "name": self.name,
                "pattern": PRETOKENIZE_PATTERN,
                "eos_token": self.eos_token,
                "eos_token_id": self.eos_token_id,
                "vocab_size": len(self),
                "merges": [list(m) for m in self.merges],
            }, f)
        return path

    @classmethod
    def from_pretrained(cls, directory):
        path = directory if directory.endswith(".json") else os.path.join(directory, "legal_bpe_tokenizer.json")
        with open(path) as f:
            data = json.load(f)
        assert data["pattern"] == PRETOKENIZE_PATTERN, "tokenizer was saved with a different pre-tokenisation pattern"
        tok = cls(data["merges"], name=data.get("name", "legal-bpe"))
        assert tok.eos_token_id == data["eos_token_id"]
        return tok

TOKENIZER_DIR = None
if TOKEN_CACHE_DIR:
    for _d in [os.path.join(TOKEN_CACHE_DIR, "tokenizer"), TOKEN_CACHE_DIR]:
        if os.path.exists(os.path.join(_d, "legal_bpe_tokenizer.json")):
            TOKENIZER_DIR = _d
            break
if not TOKENIZER_DIR:
    raise FileNotFoundError(
        "Own BPE tokenizer not found. Run CS781_Phase1_Tokenizer_OwnBPE.ipynb, publish its cs781_tokens "
        "output as a Kaggle dataset, add it as input and set TOKEN_CACHE_DIR to its path.")
tokenizer = LegalBPETokenizer.from_pretrained(TOKENIZER_DIR)
TOKENIZER_NAME = tokenizer.name
EOS_ID = tokenizer.eos_token_id
VOCAB_SIZE = len(tokenizer)
VOCAB_SIZE_PADDED = ((VOCAB_SIZE + 63) // 64) * 64
assert VOCAB_SIZE_PADDED < 2**16, "token ids are stored as uint16"
print("Tokenizer:", TOKENIZER_NAME, "| vocabulary size:", VOCAB_SIZE, "| padded embedding size:", VOCAB_SIZE_PADDED)

## 5. Download Training Data From Hugging Face

#### Skipped automatically when a matching tokenized cache exists.

In [ ]:
from huggingface_hub import hf_hub_download

token_meta = {
    "mix": TRAIN_MIX, "train_tokens": TRAIN_TOKENS, "val_tokens": VAL_TOKENS,
    "val_doc_prob": VAL_DOC_PROB, "seed": SEED, "tokenizer": TOKENIZER_NAME,
}

def find_token_cache():
    for d in [TOKEN_CACHE_DIR, TOKEN_DIR]:
        if not d:
            continue
        meta_path = os.path.join(d, "meta.json")
        if os.path.exists(meta_path):
            with open(meta_path) as f:
                if json.load(f) == json.loads(json.dumps(token_meta)):
                    return d
    return None

token_cache = find_token_cache()
train_file_paths = {}
if token_cache:
    print("Found matching tokenized cache in", token_cache, "- skipping download.")
else:
    for fname, weight in TRAIN_MIX.items():
        if weight <= 0:
            continue
        train_file_paths[fname] = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=f"{TRAIN_DIR}/{fname}",
            repo_type="dataset",
            token=HF_TOKEN,
        )
        size_gb = os.path.getsize(train_file_paths[fname]) / 1024**3
        print(f"{fname}: {train_file_paths[fname]} ({size_gb:.2f} GB)")

## 6. Create Train and Validation Tokens Set

#### Mixes the sources by `TRAIN_MIX`, splits train/val **by document**, adds an EOS separator between documents, and stores the tokens as `uint16`.

In [ ]:
import gc

TOKENIZE_BATCH_DOCS = 64
LARGE_FILE_GB = 2.0
BYTES_PER_TOKEN = 4.0

def load_records(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        data = list(data.values())
    return data

def record_text(record):
    if isinstance(record, dict):
        return record.get("text") or ""
    return str(record)

def iter_documents(path, rng, needed_tokens):
    size = os.path.getsize(path)
    if size < LARGE_FILE_GB * 1024**3:
        records = load_records(path)
        order = list(range(len(records)))
        rng.shuffle(order)
        print(f"  loaded {len(records):,} documents into memory")
        for i in order:
            yield record_text(records[i])
    else:
        try:
            import ijson
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ijson"], check=True)
            import ijson
        keep_prob = min(1.0, 2.0 * needed_tokens / (size / BYTES_PER_TOKEN))
        print(f"  streaming {size / 1024**3:.1f} GB file, keeping each document with p={keep_prob:.3f}")
        with open(path, "rb") as f:
            head = f.read(4096).lstrip()[:1]
            f.seek(0)
            stream = ijson.items(f, "item") if head == b"[" else (v for _, v in ijson.kvitems(f, ""))
            for record in stream:
                if rng.random() < keep_prob:
                    yield record_text(record)

def batched(iterable, n):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == n:
            yield batch
            batch = []
    if batch:
        yield batch

if token_cache:
    train_tokens = np.load(os.path.join(token_cache, "train_tokens.npy"))
    val_tokens = np.load(os.path.join(token_cache, "val_tokens.npy"))
else:
    rng = random.Random(SEED)
    total_weight = sum(w for w in TRAIN_MIX.values() if w > 0)
    train_chunks, val_chunks = [], []
    for fname, weight in TRAIN_MIX.items():
        if weight <= 0:
            continue
        train_quota = int(TRAIN_TOKENS * weight / total_weight)
        val_quota = int(VAL_TOKENS * weight / total_weight)
        t0 = time.time()
        print(fname)
        documents = iter_documents(train_file_paths[fname], rng, train_quota + val_quota)
        n_train = n_val = n_docs = 0
        for texts in batched(documents, TOKENIZE_BATCH_DOCS):
            for ids in tokenizer(texts, add_special_tokens=False)["input_ids"]:
                if not ids:
                    continue
                n_docs += 1
                ids.append(EOS_ID)
                if rng.random() < VAL_DOC_PROB:
                    if n_val < val_quota:
                        ids = ids[:val_quota - n_val]
                        val_chunks.append(np.asarray(ids, dtype=np.uint16))
                        n_val += len(ids)
                elif n_train < train_quota:
                    ids = ids[:train_quota - n_train]
                    train_chunks.append(np.asarray(ids, dtype=np.uint16))
                    n_train += len(ids)
            if n_train >= train_quota and n_val >= val_quota:
                break
        documents.close()
        print(f"  {n_docs:,} docs used | train {n_train:,} / val {n_val:,} tokens | {time.time() - t0:.0f}s")
        if n_train < train_quota:
            print(f"  note: source exhausted before its train quota of {train_quota:,}")
        del documents
        gc.collect()

    train_tokens = np.concatenate(train_chunks)
    val_tokens = np.concatenate(val_chunks)
    del train_chunks, val_chunks
    gc.collect()

    os.makedirs(TOKEN_DIR, exist_ok=True)
    np.save(os.path.join(TOKEN_DIR, "train_tokens.npy"), train_tokens)
    np.save(os.path.join(TOKEN_DIR, "val_tokens.npy"), val_tokens)
    with open(os.path.join(TOKEN_DIR, "meta.json"), "w") as f:
        json.dump(token_meta, f, indent=2)
    print("Saved token cache to", TOKEN_DIR)

print(f"Train tokens: {len(train_tokens):,}")
print(f"Validation tokens: {len(val_tokens):,}")

## 7. Create Token Blocks

In [ ]:
def create_blocks(tokens, block_size):
    n_blocks = len(tokens) // block_size
    return tokens[: n_blocks * block_size].reshape(n_blocks, block_size)

train_blocks = create_blocks(train_tokens, TRAIN_BLOCK_SIZE)
val_blocks = create_blocks(val_tokens, TRAIN_BLOCK_SIZE)

print("Training sequences:", len(train_blocks))
print("Validation sequences:", len(val_blocks))

## 8. Create Datasets

In [ ]:
import torch

class TokenBlockDataset(torch.utils.data.Dataset):
    def __init__(self, blocks):
        self.blocks = blocks

    def __len__(self):
        return self.blocks.shape[0]

    def __getitem__(self, idx):
        return torch.from_numpy(self.blocks[idx].astype(np.int64))

    def get_batch(self, indices):
        return torch.from_numpy(self.blocks[np.sort(np.asarray(indices))].astype(np.int64))

train_dataset = TokenBlockDataset(train_blocks)
val_dataset = TokenBlockDataset(val_blocks)

print(f"train_dataset: {len(train_dataset):,} blocks x {TRAIN_BLOCK_SIZE}")
print(f"val_dataset:   {len(val_dataset):,} blocks x {TRAIN_BLOCK_SIZE}")

## 9. Define The Model

#### 9a. Shared building blocks

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_positions, theta=10_000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv_freq)
        self.register_buffer("cos_cached", torch.cat([freqs.cos(), freqs.cos()], dim=-1), persistent=False)
        self.register_buffer("sin_cached", torch.cat([freqs.sin(), freqs.sin()], dim=-1), persistent=False)

    def forward(self, x, offset=0):
        T = x.size(-2)
        cos = self.cos_cached[offset:offset + T].to(x.dtype)
        sin = self.sin_cached[offset:offset + T].to(x.dtype)
        x1, x2 = x.chunk(2, dim=-1)
        return x * cos + torch.cat((-x2, x1), dim=-1) * sin

class SelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.0):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout

    def forward(self, x, rope, attn_mask=None, is_causal=True, cache=None, offset=0):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k = rope(q, offset), rope(k, offset)
        if cache is not None:
            if "k" in cache:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            is_causal=is_causal and attn_mask is None,
            dropout_p=self.dropout if self.training else 0.0,
        )
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, n_embd, hidden):
        super().__init__()
        self.w_gate_up = nn.Linear(n_embd, 2 * hidden, bias=False)
        self.w_down = nn.Linear(hidden, n_embd, bias=False)

    def forward(self, x):
        gate, up = self.w_gate_up(x).chunk(2, dim=-1)
        return self.w_down(F.silu(gate) * up)

class DecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.attn = SelfAttention(n_embd, n_head, dropout)
        self.norm2 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, rope, attn_mask=None, is_causal=True, cache=None, offset=0):
        x = x + self.drop(self.attn(self.norm1(x), rope, attn_mask, is_causal, cache, offset))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x

def init_weights(model, n_layers, std=0.02):
    for module in model.modules():
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
    for name, p in model.named_parameters():
        if name.endswith("proj.weight") or name.endswith("w_down.weight"):
            nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * n_layers))

def lm_loss(lm_head, hidden, input_ids, loss_start=0):
    start = max(int(loss_start), 1)
    logits = lm_head(hidden[:, start - 1:-1])
    targets = input_ids[:, start:]
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

#### 9b. Architecture

In [ ]:
class MultiScaleRetention(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.qkvg = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        gammas = 1.0 - 2.0 ** (-5.0 - torch.arange(n_head, dtype=torch.float32))
        self.register_buffer("log_gamma", torch.log(gammas), persistent=False)

    def _decay_matrix(self, T, device):
        idx = torch.arange(T, device=device, dtype=torch.float32)
        diff = idx[:, None] - idx[None, :]
        decay = torch.exp(diff.clamp(min=0)[None] * self.log_gamma[:, None, None])
        decay = decay.masked_fill(diff[None] < 0, 0.0)
        return decay / decay.sum(dim=-1, keepdim=True).sqrt()

    def forward(self, x, rope, state=None, offset=0):
        B, T, C = x.shape
        q, k, v, g = self.qkvg(x).split(C, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k = rope(q, offset), rope(k, offset)
        k = k * self.head_dim ** -0.5

        if state is not None and "S" in state:
            with torch.autocast(device_type=x.device.type, enabled=False):
                gamma = self.log_gamma.exp().view(1, -1, 1, 1)
                state["S"] = gamma * state["S"] + k.float().transpose(-1, -2) @ v.float()
                state["scale"] = gamma * state["scale"] + 1.0
                y = ((q.float() @ state["S"]) / state["scale"].sqrt()).to(v.dtype)
        else:
            decay = self._decay_matrix(T, x.device).to(q.dtype)
            y = ((q @ k.transpose(-1, -2)) * decay) @ v
            if state is not None:
                with torch.autocast(device_type=x.device.type, enabled=False):
                    pos = torch.arange(T, device=x.device, dtype=torch.float32)
                    w = torch.exp((T - 1 - pos)[None, :] * self.log_gamma[:, None])
                    state["S"] = torch.einsum("bhtd,ht,bhte->bhde", k.float(), w, v.float())
                    state["scale"] = w.sum(dim=-1).view(1, -1, 1, 1)

        y = y.transpose(1, 2).float()
        y = y * torch.rsqrt(y.pow(2).mean(dim=-1, keepdim=True) + 1e-6)
        y = y.reshape(B, T, C).to(g.dtype)
        return self.proj(F.silu(g) * y)

class FeedForward(nn.Module):
    def __init__(self, n_embd, hidden):
        super().__init__()
        self.w_in = nn.Linear(n_embd, hidden)
        self.w_down = nn.Linear(hidden, n_embd)

    def forward(self, x):
        return self.w_down(F.gelu(self.w_in(x)))

class RetNetBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.retention = MultiScaleRetention(n_embd, n_head)
        self.norm2 = nn.LayerNorm(n_embd)
        self.ffn = FeedForward(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, rope, state=None, offset=0):
        x = x + self.drop(self.retention(self.norm1(x), rope, state, offset))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x

class RetNetLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, L, H = cfg["n_embd"], cfg["n_layer"], cfg["n_head"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.blocks = nn.ModuleList([RetNetBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(L)])
        self.norm = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, L)

    def _hidden(self, input_ids, states=None, offset=0):
        x = self.drop(self.embed(input_ids))
        for i, block in enumerate(self.blocks):
            x = block(x, self.rope, None if states is None else states[i], offset)
        return self.norm(x)

    def forward(self, input_ids, prefix_len=0):
        return lm_loss(self.lm_head, self._hidden(input_ids), input_ids, prefix_len)

    @torch.no_grad()
    def prefill(self, prompt_ids):
        states = [dict() for _ in self.blocks]
        h = self._hidden(prompt_ids, states, offset=0)
        return self.lm_head(h[:, -1]), {"states": states, "pos": prompt_ids.size(1)}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self._hidden(token_ids[:, None], state["states"], offset=state["pos"])
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        ids = torch.cat([prompt_ids, cont_ids], dim=1)
        return self.lm_head(self._hidden(ids)[:, prompt_ids.size(1) - 1:-1])

MODEL_CLASS = RetNetLM
model_config = dict(
    arch=ARCH_NAME,
    vocab_size=VOCAB_SIZE_PADDED,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    ffn_hidden=FFN_HIDDEN,
    rope_theta=ROPE_THETA,
    dropout=DROPOUT,
    max_positions=MODEL_CONTEXT_LENGTH + MAX_GENERATION_TOKENS,
    bos_token_id=EOS_ID,
    eos_token_id=EOS_ID,
)
TINY_TEST_OVERRIDES = dict(vocab_size=97, n_embd=32, n_layer=2, n_head=4, ffn_hidden=64, max_positions=64)

#### 9c. Instantiate

In [ ]:
model = MODEL_CLASS(model_config)

model_size = sum(p.numel() for p in model.parameters())
print(model.__class__.__name__)
print(json.dumps(model_config, indent=2))
print(f"Model size: {model_size / 1e6:.2f}M parameters")
if not DRY_RUN:
    assert 50e6 <= model_size <= 300e6, "Phase-1 models must have between 50M and 300M parameters"

#### 9d. Sanity check: incremental decoding equals the parallel forward pass

In [ ]:
def check_incremental_decoding(model_cls, cfg, prompt_len=12, cont_len=6, atol=1e-4):
    torch.manual_seed(0)
    tiny = model_cls(cfg).eval()
    V = cfg["vocab_size"]
    prompt = torch.randint(0, V, (2, prompt_len))
    cont = torch.randint(0, V, (2, cont_len))
    with torch.no_grad():
        reference = tiny.teacher_forced_logits(prompt, cont)
        logits, state = tiny.prefill(prompt)
        incremental = [logits]
        for j in range(cont_len - 1):
            logits, state = tiny.decode_step(cont[:, j], state)
            incremental.append(logits)
        incremental = torch.stack(incremental, dim=1)
    max_diff = (reference - incremental).abs().max().item()
    assert max_diff < atol, f"incremental decoding mismatch: {max_diff}"
    return max_diff

tiny_cfg = dict(model_config)
tiny_cfg.update(TINY_TEST_OVERRIDES)
print("max |parallel - incremental| logit difference:", check_incremental_decoding(MODEL_CLASS, tiny_cfg))

#### 9e. Training objective

In [ ]:
def sample_train_prefix_len(rng):
    return 0

EVAL_PREFIX_LEN = 0

## 10. Training Arguments and Trainer

In [ ]:
import contextlib
from types import SimpleNamespace
from huggingface_hub import HfApi

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpu = torch.cuda.device_count()
micro_batch = MICRO_BATCH_PER_GPU * max(1, n_gpu)
grad_accum = max(1, GLOBAL_BATCH_SEQS // micro_batch)
steps_per_epoch = len(train_dataset) // (micro_batch * grad_accum)
max_steps = steps_per_epoch * EPOCHS
assert max_steps > 0, "not enough training blocks for one optimizer step"

training_args = SimpleNamespace(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    micro_batch=micro_batch,
    grad_accum=grad_accum,
    eval_batch_size=micro_batch,
    learning_rate=LEARNING_RATE,
    min_lr=LEARNING_RATE * MIN_LR_RATIO,
    weight_decay=WEIGHT_DECAY,
    betas=ADAM_BETAS,
    warmup_steps=min(WARMUP_STEPS, max(1, max_steps // 10)),
    max_steps=max_steps,
    grad_clip=GRAD_CLIP,
    fp16=torch.cuda.is_available(),
    eval_steps=EVAL_STEPS,
    eval_batches=EVAL_BATCHES,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    max_train_hours=MAX_TRAIN_HOURS,
    resume=RESUME,
    hf_ckpt_repo=HF_CKPT_REPO,
    hf_backup_steps=HF_BACKUP_STEPS,
    seed=SEED,
)
tokens_per_step = micro_batch * grad_accum * TRAIN_BLOCK_SIZE
print(f"GPUs: {n_gpu} | micro batch: {micro_batch} | grad accum: {grad_accum} | tokens/step: {tokens_per_step:,}")
print(f"Optimizer steps: {max_steps:,} ({EPOCHS} epoch(s), {max_steps * tokens_per_step:,} tokens)")

class LegalLMTrainer:

    def __init__(self, model, args, train_dataset, eval_dataset, sample_prefix_len, eval_prefix_len, cont_prefix_len):
        self.args = args
        self.model = model.to(device)
        self.runner = nn.DataParallel(self.model) if n_gpu > 1 else self.model
        self.train_dataset, self.eval_dataset = train_dataset, eval_dataset
        self.sample_prefix_len = sample_prefix_len
        self.eval_prefix_len, self.cont_prefix_len = eval_prefix_len, cont_prefix_len

        decay = [p for p in self.model.parameters() if p.requires_grad and p.dim() >= 2]
        no_decay = [p for p in self.model.parameters() if p.requires_grad and p.dim() < 2]
        self.optimizer = torch.optim.AdamW(
            [{"params": decay, "weight_decay": args.weight_decay},
             {"params": no_decay, "weight_decay": 0.0}],
            lr=args.learning_rate, betas=args.betas, eps=1e-8,
            fused=torch.cuda.is_available(),
        )
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(self.optimizer, self._lr_lambda)
        self.scaler = torch.amp.GradScaler("cuda", enabled=args.fp16)
        self.state = SimpleNamespace(global_step=0, log_history=[], best_eval_loss=float("inf"), best_step=None)
        self.py_rng = random.Random(args.seed)
        os.makedirs(args.output_dir, exist_ok=True)
        self.last_ckpt = os.path.join(args.output_dir, "last.pt")
        self.best_ckpt = os.path.join(args.output_dir, "best_model.pt")
        self.session_tokens, self.session_seconds, self.stopped_early = 0, 0.0, False

    def _lr_lambda(self, step):
        a = self.args
        if step < a.warmup_steps:
            return (step + 1) / a.warmup_steps
        progress = min(1.0, (step - a.warmup_steps) / max(1, a.max_steps - a.warmup_steps))
        min_ratio = a.min_lr / a.learning_rate
        return min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))

    def _autocast(self):
        if self.args.fp16:
            return torch.autocast(device_type="cuda", dtype=torch.float16)
        return contextlib.nullcontext()

    def _loss(self, ids, prefix_len):
        with self._autocast():
            loss = self.runner(ids, prefix_len=prefix_len)
        return loss.float().mean()

    @torch.no_grad()
    def evaluate(self, max_batches=None):
        self.model.eval()
        bs = self.args.eval_batch_size
        N = len(self.eval_dataset)
        n = N if max_batches is None else min(N, max_batches * bs)
        order = np.arange(N) if n == N else np.linspace(0, N - 1, n).round().astype(np.int64)
        main_sum = cont_sum = 0.0
        count = 0
        for s in range(0, n, bs):
            ids = self.eval_dataset.get_batch(order[s:s + bs]).to(device)
            b = ids.size(0)
            main = self._loss(ids, self.eval_prefix_len).item()
            cont = main if self.eval_prefix_len == self.cont_prefix_len else self._loss(ids, self.cont_prefix_len).item()
            main_sum += main * b
            cont_sum += cont * b
            count += b
        self.model.train()
        eval_loss, cont_loss = main_sum / count, cont_sum / count
        return {
            "eval_loss": eval_loss,
            "eval_perplexity": math.exp(min(eval_loss, 20)),
            "eval_cont_loss": cont_loss,
            "eval_cont_perplexity": math.exp(min(cont_loss, 20)),
        }

    def _save_checkpoint(self):
        ckpt = {
            "model": self.model.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "scheduler": self.scheduler.state_dict(),
            "scaler": self.scaler.state_dict(),
            "state": vars(self.state),
            "py_rng": self.py_rng.getstate(),
        }
        tmp = self.last_ckpt + ".tmp"
        torch.save(ckpt, tmp)
        os.replace(tmp, self.last_ckpt)

    def _backup_to_hub(self):
        if not self.args.hf_ckpt_repo:
            return
        try:
            api = HfApi(token=HF_TOKEN)
            api.create_repo(self.args.hf_ckpt_repo, private=True, exist_ok=True)
            for path in [self.last_ckpt, self.best_ckpt]:
                if os.path.exists(path):
                    api.upload_file(path_or_fileobj=path, path_in_repo=os.path.basename(path),
                                    repo_id=self.args.hf_ckpt_repo)
            print(f"  backed up checkpoints to {self.args.hf_ckpt_repo}")
        except Exception as e:
            print("  HF backup failed:", e)

    def _maybe_resume(self):
        if not self.args.resume:
            return
        if not os.path.exists(self.last_ckpt) and self.args.hf_ckpt_repo:
            for fname in ["last.pt", "best_model.pt"]:
                try:
                    path = hf_hub_download(self.args.hf_ckpt_repo, fname, token=HF_TOKEN)
                    shutil.copy(path, os.path.join(self.args.output_dir, fname))
                except Exception as e:
                    print(f"  no {fname} on the Hub ({type(e).__name__})")
        if os.path.exists(self.last_ckpt):
            ckpt = torch.load(self.last_ckpt, map_location="cpu", weights_only=False)
            self.model.load_state_dict(ckpt["model"])
            self.optimizer.load_state_dict(ckpt["optimizer"])
            self.scheduler.load_state_dict(ckpt["scheduler"])
            self.scaler.load_state_dict(ckpt["scaler"])
            self.state = SimpleNamespace(**ckpt["state"])
            self.py_rng.setstate(ckpt["py_rng"])
            print(f"Resumed from step {self.state.global_step:,}")

    def train(self):
        a = self.args
        self._maybe_resume()
        self.model.train()
        steps_per_epoch = a.max_steps // a.num_train_epochs
        tokens_per_step = a.micro_batch * a.grad_accum * TRAIN_BLOCK_SIZE
        start = time.time()
        perm_epoch, perm = None, None
        log_loss, log_count = 0.0, 0

        while self.state.global_step < a.max_steps:
            epoch, within = divmod(self.state.global_step, steps_per_epoch)
            if epoch != perm_epoch:
                perm = np.random.default_rng(a.seed + epoch).permutation(len(self.train_dataset))
                perm_epoch = epoch
            prefix_len = self.sample_prefix_len(self.py_rng)

            self.optimizer.zero_grad(set_to_none=True)
            step_loss = 0.0
            for micro in range(a.grad_accum):
                s = (within * a.grad_accum + micro) * a.micro_batch
                ids = self.train_dataset.get_batch(perm[s:s + a.micro_batch]).to(device, non_blocking=True)
                loss = self._loss(ids, prefix_len)
                self.scaler.scale(loss / a.grad_accum).backward()
                step_loss += loss.detach()
            self.scaler.unscale_(self.optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), a.grad_clip)
            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.scheduler.step()

            self.state.global_step += 1
            step = self.state.global_step
            self.session_tokens += tokens_per_step
            log_loss = log_loss + step_loss / a.grad_accum
            log_count += 1

            if step % a.logging_steps == 0 or step == a.max_steps:
                avg = float(log_loss) / log_count
                elapsed = time.time() - start
                tps = self.session_tokens / elapsed
                eta_h = (a.max_steps - step) * tokens_per_step / tps / 3600
                entry = {"step": step, "epoch": step / steps_per_epoch, "loss": avg,
                         "learning_rate": self.scheduler.get_last_lr()[0], "grad_norm": float(grad_norm)}
                self.state.log_history.append(entry)
                print(f"step {step:>6}/{a.max_steps} | loss {avg:.4f} | lr {entry['learning_rate']:.2e} | "
                      f"gnorm {entry['grad_norm']:.2f} | {tps:,.0f} tok/s | ETA {eta_h:.2f} h")
                log_loss, log_count = 0.0, 0

            if step % a.eval_steps == 0 or step == a.max_steps:
                metrics = self.evaluate(a.eval_batches)
                self.state.log_history.append({"step": step, **metrics})
                print(f"  eval | loss {metrics['eval_loss']:.4f} (ppl {metrics['eval_perplexity']:.2f}) | "
                      f"cont loss {metrics['eval_cont_loss']:.4f} (ppl {metrics['eval_cont_perplexity']:.2f})")
                if metrics["eval_loss"] < self.state.best_eval_loss:
                    self.state.best_eval_loss, self.state.best_step = metrics["eval_loss"], step
                    torch.save(self.model.state_dict(), self.best_ckpt)
                    print(f"  new best checkpoint at step {step}")

            if step % a.save_steps == 0:
                self._save_checkpoint()
            if a.hf_ckpt_repo and step % a.hf_backup_steps == 0:
                self._backup_to_hub()
            if time.time() - start > a.max_train_hours * 3600 and step < a.max_steps:
                print(f"Time budget of {a.max_train_hours} h reached at step {step}. "
                      "Checkpoint saved - re-run with RESUME=True to continue.")
                self.stopped_early = True
                break

        self._save_checkpoint()
        self._backup_to_hub()
        self.session_seconds = time.time() - start
        return self.state

## 11. Training

In [ ]:
trainer = LegalLMTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    sample_prefix_len=sample_train_prefix_len,
    eval_prefix_len=EVAL_PREFIX_LEN,
    cont_prefix_len=CONT_EVAL_PREFIX,
)

start_time = time.time()

trainer.train()

training_time = (time.time() - start_time)
tokens_per_second = trainer.session_tokens / max(trainer.session_seconds, 1e-9)

print(f"Training time (this session): {training_time / 3600:.2f} hours")
print(f"Training speed: {tokens_per_second:,.0f} tokens/sec")
print(f"Steps completed: {trainer.state.global_step:,}/{training_args.max_steps:,}")

## 12. Checkpoint Selection and Perplexity

In [ ]:
if SELECT_BEST_CHECKPOINT and os.path.exists(trainer.best_ckpt):
    model.load_state_dict(torch.load(trainer.best_ckpt, map_location=device))
    print(f"Using best checkpoint (step {trainer.state.best_step}, subset eval loss {trainer.state.best_eval_loss:.4f})")
else:
    print("Using the last checkpoint")

eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)
print(f"Validation loss: {eval_loss:.6f}")
print(f"Validation perplexity: {perplexity:.4f}")
print(f"Continuation loss (last {TRAIN_BLOCK_SIZE - CONT_EVAL_PREFIX} tokens | first {CONT_EVAL_PREFIX}): "
      f"{eval_results['eval_cont_loss']:.6f} (ppl {eval_results['eval_cont_perplexity']:.4f})")

metrics = {
    "arch": ARCH_NAME,
    "parameters": model_size,
    "train_tokens": int(len(train_tokens)),
    "steps": trainer.state.global_step,
    "tokens_per_second": tokens_per_second,
    "best_step": trainer.state.best_step,
    **eval_results,
}
with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved to", METRICS_PATH)

## 13. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history

train_steps = []
train_loss = []
eval_steps = []
eval_loss = []
cont_loss = []

for log in logs:
    if ("loss" in log and "step" in log):
        train_steps.append(log["step"])
        train_loss.append(log["loss"])
    if ("eval_loss" in log and "step" in log):
        eval_steps.append(log["step"])
        eval_loss.append(log["eval_loss"])
        cont_loss.append(log["eval_cont_loss"])

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_loss, label="Training Loss")
plt.plot(eval_steps, eval_loss, marker="o", label="Validation Loss")
if EVAL_PREFIX_LEN != CONT_EVAL_PREFIX:
    plt.plot(eval_steps, cont_loss, marker="s", label="Validation Continuation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title(f"Phase 1 Training and Validation Loss ({ARCH_NAME})")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()

print("Loss plot saved to:")
print(LOSS_PLOT_PATH)

## 14. Save Final Model to Kaggle Output

In [ ]:
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
torch.save(model.state_dict(), os.path.join(MODEL_OUTPUT_DIR, "pytorch_model.bin"))
with open(os.path.join(MODEL_OUTPUT_DIR, "config.json"), "w") as f:
    json.dump(model_config, f, indent=2)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print("Final model saved to:")
print(MODEL_OUTPUT_DIR)

## 15. Push Model to Hugging Face Hub

## 16. Load Test Data from HuggingFace

In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)

## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

## 18. Load Final Model for Generation

In [ ]:
for name in ["trainer", "model"]:
    if name in globals():
        del globals()[name]
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generation_tokenizer = LegalBPETokenizer.from_pretrained(MODEL_OUTPUT_DIR)
with open(os.path.join(MODEL_OUTPUT_DIR, "config.json")) as f:
    generation_config = json.load(f)
generation_model = MODEL_CLASS(generation_config)
generation_model.load_state_dict(
    torch.load(os.path.join(MODEL_OUTPUT_DIR, "pytorch_model.bin"), map_location="cpu")
)
generation_model = generation_model.to(device)
generation_model.eval()

print("Device:", device)
print(f"Parameters: {sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M")

## 19. Generate 128 Words
#### (words, not tokens). Sampling with temperature / top-p / top-k / repetition penalty; generation stops early once every sequence in the batch has more than 128 words.

In [ ]:
import contextlib

REAL_VOCAB = len(generation_tokenizer)
BANNED_TOKEN_IDS = [generation_tokenizer.eos_token_id]

def sample_next_token(logits, generated):
    logits = logits.float()
    if logits.size(-1) > REAL_VOCAB:
        logits[:, REAL_VOCAB:] = float("-inf")
    logits[:, BANNED_TOKEN_IDS] = float("-inf")
    if generated is not None and REPETITION_PENALTY != 1.0:
        prev = logits.gather(1, generated)
        prev = torch.where(prev > 0, prev / REPETITION_PENALTY, prev * REPETITION_PENALTY)
        logits = logits.scatter(1, generated, prev)
    if TEMPERATURE <= 0:
        return logits.argmax(dim=-1)
    logits = logits / TEMPERATURE
    if TOP_K > 0:
        kth = torch.topk(logits, TOP_K, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if TOP_P < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
        probs = sorted_logits.softmax(dim=-1)
        remove = probs.cumsum(dim=-1) - probs > TOP_P
        sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(1, sorted_idx, sorted_logits)
    return torch.multinomial(logits.softmax(dim=-1), num_samples=1).squeeze(1)

def enough_words(generated, tokenizer, target_words):
    texts = tokenizer.batch_decode(generated.tolist(), skip_special_tokens=True)
    return all(len(t.split()) > target_words for t in texts)

@torch.inference_mode()
def generate_token_ids(model, prompt_ids, tokenizer, target_words):
    amp = torch.autocast(device_type="cuda", dtype=torch.float16) if device.type == "cuda" else contextlib.nullcontext()
    with amp:
        logits, state = model.prefill(prompt_ids)
        generated = []
        for step in range(MAX_GENERATION_TOKENS):
            prev = torch.stack(generated, dim=1) if generated else None
            next_token = sample_next_token(logits, prev)
            generated.append(next_token)
            if (step + 1) % 16 == 0 and enough_words(torch.stack(generated, dim=1), tokenizer, target_words):
                break
            if step + 1 < MAX_GENERATION_TOKENS:
                logits, state = model.decode_step(next_token, state)
    return torch.stack(generated, dim=1)

def prepare_prompt_ids(text, tokenizer):
    ids = tokenizer.encode(text, add_special_tokens=False) or [tokenizer.eos_token_id]
    return ids[-PROMPT_TOKENS:]

def generate_batch(prompt_id_lists, model, tokenizer, device, target_words=128):
    input_ids = torch.tensor(prompt_id_lists, dtype=torch.long, device=device)
    generated = generate_token_ids(model, input_ids, tokenizer, target_words)

    predictions = []
    for row in generated.tolist():
        generated_text = tokenizer.decode(row, skip_special_tokens=True)
        generated_words = generated_text.split()[:target_words]
        predictions.append(" ".join(generated_words))
    return predictions

## 20. Generate Predictions for the Test Set

In [ ]:
from collections import defaultdict

predictions = {}
torch.manual_seed(SEED)

items = list(test_data.items())
if MAX_TEST_EXAMPLES:
    items = items[:MAX_TEST_EXAMPLES]
total_examples = len(items)
print(f"Total test examples: {total_examples}")

buckets = defaultdict(list)
for example_id, record in items:
    ids = prepare_prompt_ids(record["text"], generation_tokenizer)
    buckets[len(ids)].append((str(example_id), ids))
batches = []
for length in sorted(buckets, reverse=True):
    group = buckets[length]
    for start in range(0, len(group), GENERATION_BATCH_SIZE):
        batches.append(group[start:start + GENERATION_BATCH_SIZE])
print(f"{len(batches)} batches ({len(buckets)} distinct prompt lengths)")

t0 = time.time()
completed = 0
for batch_index, batch in enumerate(batches):
    batch_predictions = generate_batch(
        prompt_id_lists=[ids for _, ids in batch],
        model=generation_model,
        tokenizer=generation_tokenizer,
        device=device,
        target_words=TARGET_GENERATION_WORDS,
    )
    for (example_id, _), prediction in zip(batch, batch_predictions):
        predictions[example_id] = {"text": prediction}
    completed += len(batch)
    if batch_index % 5 == 0 or completed == total_examples:
        print(f"Completed {completed}/{total_examples} ({100 * completed / total_examples:.1f}%) | {time.time() - t0:.0f}s")

missing = set(str(k) for k, _ in items) - set(predictions)
assert not missing, f"{len(missing)} test ids have no prediction"
short = sum(len(p["text"].split()) < TARGET_GENERATION_WORDS for p in predictions.values())
print(f"Predictions with fewer than {TARGET_GENERATION_WORDS} words: {short}")
first_id = items[0][0]
print("\nSample prompt end:", " ".join(test_data[first_id]["text"].split()[-40:]))
print("\nSample continuation:", predictions[str(first_id)]["text"])

## 21. Save prediction.json

In [ ]:
if len(predictions) != len(test_data):
    print(f"WARNING: {len(predictions)} predictions for {len(test_data)} test examples "
          "(MAX_TEST_EXAMPLES is set) - this file is not a valid submission.")

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("prediction.json saved:")
print(PREDICTION_PATH)

## 22. Create CodaBench Submission ZIP

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("Codabench submission created:")
print(SUBMISSION_ZIP)